# 진자 엔코더: 들어 올렸다가 놓는 자유응답 측정

이 노트북은 rotary arm을 구동하지 않고 진자 엔코더만 조회하여 데이터를 저장한 뒤 plot한다. 사람이 진자를 손으로 들어 올리고, 준비가 되면 기록을 시작한 다음 countdown에 맞춰 놓는다.

- 펌웨어: `PendulumController/PendulumController.ino`
- 통신: newline-delimited JSON, 500000 baud
- 진자 각도: `observation[0]`, 원시 범위 $[0,360)$ degree
- 분해능: 1200 count/rev, 즉 $0.3^\circ$/count

> 안전: 아래 코드는 연결 직후 `CMD_HARD_STOP` 한 번과 `CMD_QUERY`만 보낸다. 모터 이동 명령은 보내지 않는다. 그래도 장치 주변을 비우고, rotary arm이 확실히 정지했는지 확인한다.

## 0. 설치와 설정

저장소 루트에서 `python -m pip install -r requirements.txt`를 실행한다. Windows에서는 포트가 자동 탐지되지 않으면 `SERIAL_PORT='COM5'`처럼 지정한다. Linux라면 `/dev/ttyACM0` 형태이다.

이 노트북은 먼저 진자를 완전히 아래로 늘어뜨린 상태에서 원형 평균을 계산해 아래쪽 영점을 정한다. 그 다음 사람이 진자를 들어 올리고 Enter를 누르면 countdown 후 기록한다.

In [ ]:
from pathlib import Path
from datetime import datetime
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

from control_comms import ControlComms, DebugLevel, StatusCode, find_board_port

SERIAL_PORT = None       # None이면 ST-Link 자동 탐지
BAUD = 500_000
TIMEOUT_S = 0.2
CALIBRATION_S = 2.0
RECORD_S = 20.0
COUNTDOWN_S = 3
CMD_HARD_STOP = 5
CMD_QUERY = 6

## 1. 보조 함수

엔코더 값이 $359.9^\circ$와 $0.1^\circ$ 사이를 오갈 때 산술평균은 잘못된 $180^\circ$를 줄 수 있다. 따라서 아래쪽 영점은 원형 평균으로 계산한다. 영점을 뺀 값은

$$\theta=\big((q-q_{\rm down}+180)\bmod360\big)-180$$

으로 $[-180,180)$에 감싼다. 각 샘플에는 호스트의 monotonic timestamp와 MCU millisecond timestamp를 모두 저장한다.

In [ ]:
def wrap_deg(angle):
    return (np.asarray(angle) + 180.0) % 360.0 - 180.0

def circular_mean_deg(values):
    radians = np.deg2rad(values)
    return np.rad2deg(np.arctan2(np.sin(radians).mean(), np.cos(radians).mean())) % 360.0

def query_sample(ctrl):
    query_ns = time.perf_counter_ns()
    response = ctrl.step(CMD_QUERY, [0.0])
    received_ns = time.perf_counter_ns()
    if response is None:
        return None
    status, mcu_ms, terminated, obs = response
    if len(obs) < 4:
        raise RuntimeError(f'{len(obs)} observations received; expected 4. Flash this repository firmware.')
    return dict(host_query_ns=query_ns, host_received_ns=received_ns, mcu_ms=mcu_ms,
                status=status, terminated=terminated, pendulum_raw_deg=obs[0],
                rotor_deg=obs[1], motor_speed_pps=obs[2], l6474_status=int(obs[3]))

## 2. 연결

아래 셀은 포트를 열고 이전 motor motion을 hard stop한다. 응답의 observation 개수가 4가 아니면 다른 firmware가 올라가 있는 것이므로 이 저장소의 firmware를 다시 flash한다.

In [ ]:
port = find_board_port(SERIAL_PORT)
ctrl = ControlComms(timeout=TIMEOUT_S, debug_level=DebugLevel.DEBUG_ERROR)
if ctrl.connect(port, BAUD) is not StatusCode.OK:
    raise RuntimeError(f'Could not open {port}')
ctrl.step(CMD_HARD_STOP, [0.0])
test = query_sample(ctrl)
print('Connected:', port, test)

## 3. 아래쪽 영점 보정

진자를 손대지 말고 완전히 아래로 늘어뜨린다. 아래 셀을 실행하는 동안 정지 상태를 유지한다. 보정 중 최대 잔차가 엔코더 한두 count보다 훨씬 크면 진동이 가라앉은 뒤 다시 실행한다.

In [ ]:
calibration = []
deadline = time.perf_counter() + CALIBRATION_S
while time.perf_counter() < deadline:
    sample = query_sample(ctrl)
    if sample is not None:
        calibration.append(sample['pendulum_raw_deg'])
if len(calibration) < 3:
    raise RuntimeError('Too few calibration samples; check serial connection.')
down_reference_deg = float(circular_mean_deg(calibration))
residual = wrap_deg(np.asarray(calibration)-down_reference_deg)
print(f'down reference={down_reference_deg:.3f} deg, samples={len(calibration)}, '
      f'peak residual={np.max(np.abs(residual)):.3f} deg')

## 4. 들어 올리고 놓기: 데이터 기록

1. 셀을 실행한다.
2. 진자를 원하는 각도까지 손으로 들어 올려 정지시킨다.
3. Enter를 누른다.
4. `RELEASE` 표시에 맞춰 미는 힘 없이 손을 뗀다.

Jupyter 출력과 실제 손 동작에는 사람 반응 지연이 있으므로 CSV의 $t=0$은 정확한 물리 release 순간이 아닐 수 있다. 정밀 비교에서는 첫 유의미한 속도 또는 peak를 이용해 time origin을 사후 보정한다. `Ctrl+C`로 중단해도 `finally`에서 motor hard stop과 port close를 수행한다.

In [ ]:
rows = []
input('Lift and hold the pendulum, then press Enter to arm recording... ')
for remaining in range(COUNTDOWN_S, 0, -1):
    print(f'Release in {remaining}...', flush=True)
    time.sleep(1.0)
print('RELEASE / recording', flush=True)
start_ns = time.perf_counter_ns()
try:
    while (time.perf_counter_ns()-start_ns)*1e-9 < RECORD_S:
        sample = query_sample(ctrl)
        if sample is None:
            continue
        sample['time_s'] = (sample['host_received_ns']-start_ns)*1e-9
        sample['pendulum_deg'] = float(wrap_deg(sample['pendulum_raw_deg']-down_reference_deg))
        rows.append(sample)
finally:
    try:
        ctrl.step(CMD_HARD_STOP, [0.0])
    finally:
        ctrl.close()
print(f'Acquired {len(rows)} samples.')

## 5. CSV 저장

첫 줄에는 영점/포트 정보를 comment로, 그 아래에는 표준 CSV header와 samples를 저장한다. 원시각도도 보존하므로 영점을 나중에 다시 계산할 수 있다.

In [ ]:
if not rows:
    raise RuntimeError('No data to save.')
df = pd.DataFrame(rows)
column_order = ['time_s', 'host_query_ns', 'host_received_ns', 'mcu_ms', 'status', 'terminated',
                'pendulum_raw_deg', 'pendulum_deg', 'rotor_deg', 'motor_speed_pps', 'l6474_status']
df = df[column_order]
Path('data').mkdir(exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = Path('data') / f'pendulum_release_{stamp}.csv'
with csv_path.open('w', encoding='utf-8', newline='') as handle:
    handle.write(f'# down_reference_deg={down_reference_deg:.9f},port={port},baud={BAUD}\n')
    df.to_csv(handle, index=False)
rate = (len(df)-1)/(df.time_s.iloc[-1]-df.time_s.iloc[0]) if len(df)>1 else np.nan
print(f'Saved {len(df)} samples, mean rate={rate:.1f} samples/s -> {csv_path}')
df.head()

## 6. 사후 plot과 기본 품질 확인

각도와 샘플 간격을 함께 본다. USB/OS scheduling 때문에 $\Delta t$에 간헐적인 spike가 있을 수 있다. 모델 비교에는 실제 `time_s`를 그대로 사용한다. `rotor_deg`와 `motor_speed_pps`가 변한다면 rotary arm이 정말 고정되어 있었는지 확인한다.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
axes[0].plot(df.time_s, df.pendulum_deg, lw=1.2)
axes[0].set_ylabel('Pendulum (deg)'); axes[0].axhline(0, color='k', lw=.7)
axes[1].plot(df.time_s, df.rotor_deg, label='rotor deg')
axes[1].plot(df.time_s, df.motor_speed_pps, label='motor speed pps')
axes[1].set_ylabel('Rotary state'); axes[1].legend()
dt_ms = np.diff(df.time_s, prepend=np.nan)*1000
axes[2].plot(df.time_s, dt_ms, lw=.8)
axes[2].set(xlabel='Time (s)', ylabel='Sample interval (ms)')
for ax in axes: ax.grid(True, alpha=.3)
fig.tight_layout(); plt.show()
print(df[['pendulum_deg', 'rotor_deg', 'motor_speed_pps']].describe())

## 7. 선택 사항: 주기와 선형 감쇠율의 빠른 추정

같은 부호의 양의 peaks를 검출한다. 연속 두 양의 peak 사이가 한 주기이며, 점성 선형 모델의 포락선이 $e^{-\beta t}$일 때

$$\beta\approx\frac{1}{t_{n+1}-t_n}\ln\left|\frac{\theta_n}{\theta_{n+1}}\right|.$$

엔코더 양자화와 release 시점 오차 때문에 이 값은 시작 추정치일 뿐이다. 충분한 prominence와 peak 간 최소 거리를 데이터에 맞게 조정한다.

In [ ]:
median_dt = np.median(np.diff(df.time_s))
minimum_period_s = 0.3
peak_indices, properties = find_peaks(df.pendulum_deg.to_numpy(),
                                        prominence=1.0,
                                        distance=max(1, int(minimum_period_s/median_dt)))
peak_t = df.time_s.to_numpy()[peak_indices]
peak_a = df.pendulum_deg.to_numpy()[peak_indices]
if len(peak_t) >= 2:
    periods = np.diff(peak_t)
    beta_pairs = np.log(np.abs(peak_a[:-1]/peak_a[1:]))/periods
    print(f'period median={np.median(periods):.4f} s')
    print(f'beta median={np.median(beta_pairs):.4f} 1/s')
else:
    print('Need at least two positive peaks; adjust prominence/distance or record longer.')
plt.plot(df.time_s, df.pendulum_deg, label='measurement')
plt.plot(peak_t, peak_a, 'ro', label='positive peaks')
plt.xlabel('Time (s)'); plt.ylabel('Angle (deg)'); plt.grid(True, alpha=.3); plt.legend(); plt.show()

## 실시간 표시가 필요할 때

Notebook event loop/backend 차이로 실시간 갱신이 불안정할 수 있으므로 별도 프로그램을 제공한다. 저장소 루트의 terminal에서 실행한다.

```bash
python encoder_live_monitor.py --duration 20 --countdown 3
```

포트가 자동 탐지되지 않으면 `--port COM5` 또는 `--port /dev/ttyACM0`를 추가한다. 창을 닫거나 `Ctrl+C`를 누르면 기록을 멈추며, 받은 sample이 있으면 timestamped CSV를 `data/`에 저장한다.